# **Customer Lifetime Value (CLV) Modeling.**

The purpose of this notebook is to model expected future revenue of a customer for the next 60 days.

## **Load data.**

In [1]:
import os
import pandas as pd
import numpy as np

DATASETS_PATH = os.getenv("DATASETS_PATH")
DATASETS_PATH = DATASETS_PATH + "/CustomerLifeValue/EcommerceOlist/"

customers_df = pd.read_csv(DATASETS_PATH + "olist_customers_dataset.csv")

order_items_df = pd.read_csv(DATASETS_PATH + "olist_order_items_dataset.csv")

payments_df = pd.read_csv(DATASETS_PATH + "olist_order_payments_dataset.csv")

payments_df = pd.read_csv(DATASETS_PATH + "olist_order_payments_dataset.csv")

orders_df = pd.read_csv(DATASETS_PATH + "olist_orders_dataset.csv")

reviews_df = pd.read_csv(DATASETS_PATH + "olist_order_reviews_dataset.csv")

products_df = pd.read_csv(DATASETS_PATH + "olist_products_dataset.csv")

payments_agg = (
    payments_df
    .groupby("order_id")
    .agg({
        "payment_value": "sum",
        "payment_installments": "max" # Describes in how many parts the resulting sum is payed (payment behaviour feature)
    })
    .reset_index()
)

clv_df = (
    orders_df
    .merge(customers_df, on="customer_id")
    .merge(payments_agg, on="order_id")
)

reviews_small = reviews_df[
    ["order_id", "review_score"]
]

clv_df = clv_df.merge(
    reviews_small,
    on="order_id",
    how="left"
)

drop_cols = [
    "customer_id",
    "order_approved_at"
]

clv_df = clv_df[
    clv_df["order_status"] == "delivered"
]

clv_df = clv_df.drop(columns = drop_cols)

print(clv_df)


                               order_id order_status order_purchase_timestamp  \
0      e481f51cbdc54678b7cc49136f2d6af7    delivered      2017-10-02 10:56:33   
1      53cdb2fc8bc7dce0b6741e2150273451    delivered      2018-07-24 20:41:37   
2      47770eb9100c2d0c44946d9cf07ec65d    delivered      2018-08-08 08:38:49   
3      949d5b44dbf5de918fe9c16f97b45f8a    delivered      2017-11-18 19:28:06   
4      ad21c59c0840e6cb83a9ceb5573f8159    delivered      2018-02-13 21:18:39   
...                                 ...          ...                      ...   
99986  9c5dedf39a927c1b2549525ed64a053c    delivered      2017-03-09 09:54:05   
99987  63943bddc261676b46f01ca7ac2f7bd8    delivered      2018-02-06 12:58:58   
99988  83c1379a015df1e13d02aae0204711ab    delivered      2017-08-27 14:46:43   
99989  11c177c8e97725db2631073c19f07b62    delivered      2018-01-08 21:28:27   
99990  66dea50a8b16d9b4dee7af250b4be1a5    delivered      2018-03-08 20:57:30   

      order_delivered_carri

## **Calculate target column (future purchase in the next 60 days after first purchase).**

In [2]:
clv_df["order_purchase_timestamp"] = pd.to_datetime(
    clv_df["order_purchase_timestamp"]
)

clv_df = clv_df.sort_values(
    ["customer_unique_id", "order_purchase_timestamp"]
)

first_purchase = (
    clv_df.groupby("customer_unique_id")
    .first()
    .reset_index()
)

first_purchase = first_purchase[
    ["customer_unique_id",
     "order_purchase_timestamp",
     "order_delivered_customer_date",
     "order_estimated_delivery_date",
     "payment_value",
     "payment_installments",
     "review_score",
     "customer_city",
     "customer_state"]
]

first_purchase["cutoff_date"] = (
    first_purchase["order_purchase_timestamp"]
    + pd.Timedelta(days=60)
)

future_df = clv_df.merge(
    first_purchase[
        ["customer_unique_id",
         "order_purchase_timestamp",
         "cutoff_date"]
    ],
    on="customer_unique_id",
    suffixes=("", "_first")
)

future_orders = future_df[
    (
        future_df["order_purchase_timestamp"]
        > future_df["order_purchase_timestamp_first"]
    )
    &
    (
        future_df["order_purchase_timestamp"]
        <= future_df["cutoff_date"]
    )
]

future_revenue = (
    future_orders
    .groupby("customer_unique_id")["payment_value"]
    .sum()
    .reset_index()
)

future_revenue.columns = [
    "customer_unique_id",
    "future_60d_revenue"
]

model_df = first_purchase.merge(
    future_revenue,
    on="customer_unique_id",
    how="left"
)

model_df["future_60d_revenue"] = (
    model_df["future_60d_revenue"]
    .fillna(0)
)

print(model_df)

                     customer_unique_id order_purchase_timestamp  \
0      0000366f3b9a7992bf8c76cfdf3221e2      2018-05-10 10:56:27   
1      0000b849f77a49e4a4ce2b2a4ca5be3f      2018-05-07 11:11:27   
2      0000f46a3911fa3c0805444483337064      2017-03-10 21:05:03   
3      0000f6ccb0745a6a4b88665a16c9f078      2017-10-12 20:29:41   
4      0004aac84e0df4da2b147fca70cf8255      2017-11-14 19:45:42   
...                                 ...                      ...   
93352  fffcf5a5ff07b0908bd4e2dbc735a684      2017-06-08 21:00:36   
93353  fffea47cd6d3cc0a88bd621562a9d061      2017-12-10 20:07:56   
93354  ffff371b4d645b6ecea244b27531430a      2017-02-07 15:49:16   
93355  ffff5962728ec6157033ef9805bacc48      2018-05-02 15:17:41   
93356  ffffd2657e2aad2907e67c3e9daecbeb      2017-05-02 20:18:45   

      order_delivered_customer_date order_estimated_delivery_date  \
0               2018-05-16 20:48:37           2018-05-21 00:00:00   
1               2018-05-10 18:02:42          

## **Calculate delivery related features.**

In [3]:
model_df["order_delivered_customer_date"] = pd.to_datetime(
    model_df["order_delivered_customer_date"]
)

model_df["order_estimated_delivery_date"] = pd.to_datetime(
    model_df["order_estimated_delivery_date"]
)

model_df["order_purchase_timestamp"] = pd.to_datetime(
    model_df["order_purchase_timestamp"]
)

model_df["delivery_days"] = (
    model_df["order_delivered_customer_date"]
    - model_df["order_purchase_timestamp"]
).dt.days

model_df["delivery_delay_days"] = (
    model_df["order_delivered_customer_date"]
    - model_df["order_estimated_delivery_date"]
).dt.days

model_df["is_late_delivery"] = (
    model_df["delivery_delay_days"] > 0
).astype(int)

print(model_df)

                     customer_unique_id order_purchase_timestamp  \
0      0000366f3b9a7992bf8c76cfdf3221e2      2018-05-10 10:56:27   
1      0000b849f77a49e4a4ce2b2a4ca5be3f      2018-05-07 11:11:27   
2      0000f46a3911fa3c0805444483337064      2017-03-10 21:05:03   
3      0000f6ccb0745a6a4b88665a16c9f078      2017-10-12 20:29:41   
4      0004aac84e0df4da2b147fca70cf8255      2017-11-14 19:45:42   
...                                 ...                      ...   
93352  fffcf5a5ff07b0908bd4e2dbc735a684      2017-06-08 21:00:36   
93353  fffea47cd6d3cc0a88bd621562a9d061      2017-12-10 20:07:56   
93354  ffff371b4d645b6ecea244b27531430a      2017-02-07 15:49:16   
93355  ffff5962728ec6157033ef9805bacc48      2018-05-02 15:17:41   
93356  ffffd2657e2aad2907e67c3e9daecbeb      2017-05-02 20:18:45   

      order_delivered_customer_date order_estimated_delivery_date  \
0               2018-05-16 20:48:37                    2018-05-21   
1               2018-05-10 18:02:42          

## **Calculate date related features.**

In [4]:
model_df["purchase_month"] = (
    model_df["order_purchase_timestamp"].dt.month
)

model_df["purchase_weekday"] = (
    model_df["order_purchase_timestamp"].dt.dayofweek
)

## **Calculate target for retain classifier.**

In [5]:
model_df["retained_60d"] = (model_df["future_60d_revenue"] > 0).astype(int)

y_ret = model_df["retained_60d"]
y_rev = np.log1p(model_df["future_60d_revenue"])

## **Set categorical variables.**

In [6]:
for col in ["customer_city", "customer_state"]:
    model_df[col] = model_df[col].astype("category")

## **Drop unnecessary features.**

In [7]:
model_df = model_df.drop(columns = ["customer_unique_id", \
                                    "order_purchase_timestamp", \
                                    "order_delivered_customer_date",\
                                    "order_estimated_delivery_date",\
                                    "cutoff_date"                                   
                                    ])
print(model_df)

       payment_value  payment_installments  review_score       customer_city  \
0             141.90                     8           5.0             cajamar   
1              27.19                     1           4.0              osasco   
2              86.22                     8           3.0            sao jose   
3              43.62                     4           4.0               belem   
4             196.89                     6           5.0            sorocaba   
...              ...                   ...           ...                 ...   
93352        2067.42                    10           5.0             sanharo   
93353          84.58                     1           4.0    feira de santana   
93354         112.46                     1           5.0               sinop   
93355         133.69                     5           5.0  bom jesus do norte   
93356          71.56                     4           5.0         campo largo   

      customer_state         cutoff_dat

## **Split into train and test datasets.**

In [8]:
from sklearn.model_selection import train_test_split

X = model_df.drop(columns=["retained_60d", "future_60d_revenue"])

X_train, X_test, y_ret_train, y_ret_test, y_rev_train, y_rev_test = train_test_split(
    X, y_ret, y_rev,
    test_size=0.2,
    random_state=42,
    stratify=y_ret
)

## **Train retention model.**

In [9]:
from lightgbm import LGBMClassifier
from lightgbm import LGBMRegressor
from sklearn.model_selection import train_test_split

clf = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight="balanced"
)

clf.fit(
    X_train,
    y_ret_train,
    categorical_feature=["customer_city", "customer_state"]
)

retention_prob = clf.predict_proba(X_test)[:, 1]


DTypePromotionError: The DType <class 'numpy.dtypes.DateTime64DType'> could not be promoted by <class 'numpy.dtypes.Float64DType'>. This means that no common DType exists for the given inputs. For example they cannot be stored in a single array unless the dtype is `object`. The full list of DTypes is: (<class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int16DType'>, <class 'numpy.dtypes.Int8DType'>, <class 'numpy.dtypes.DateTime64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Float64DType'>, <class 'numpy.dtypes.Int64DType'>, <class 'numpy.dtypes.Int32DType'>, <class 'numpy.dtypes.Int32DType'>, <class 'numpy.dtypes.Float32DType'>)

## **Train revenue model.**

In [ ]:
reg = LGBMRegressor(
    n_estimators=3000,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    class_weight="balanced"
)

reg.fit(X_train, y_rev_train, categorical_feature=["customer_city", "customer_state"])

pred_rev = reg.predict(X_test)


## **Evaluate results.**

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

y_pred = (retention_prob >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_ret_test, y_pred))
print("Precision:", precision_score(y_ret_test, y_pred))
print("Recall:", recall_score(y_ret_test, y_pred))
print("F1:", f1_score(y_ret_test, y_pred))

print("ROC-AUC:", roc_auc_score(y_ret_test, retention_prob))
print("PR-AUC:", average_precision_score(y_ret_test, retention_prob))

from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_rev_test, pred_rev)
rmse = np.sqrt(mean_squared_error(y_rev_test, pred_rev))

print("MAE: " + str(mae))
print("RMSE: " + str(rmse))

In [ ]:
clv = retention_prob * pred_rev

from scipy.stats import spearmanr

spearmanr(y_rev_test, pred_rev)

test_results = X_test.copy()
test_results["true"] = np.expm1(y_rev_test)
test_results["clv"] = clv

top = test_results.sort_values("clv", ascending=False).head(1000)

print(top["true"].sum())